# Critical minerals: supply and demand dataset

**Inputs:** all eight `ds140-*.csv` files (cobalt, gallium, germanium, lithium, silicon, tantalum, tin, tungsten) and `MCS2026_Commodities_Data.csv`.

**Outputs:** one tidy long table (production, consumption, reserves, trade, stocks) with years as integers and masses in **metric tons**, plus a wide US summary table.

**Missing-value policy (NaN):**
1. NaN means *not reported / not applicable*. It is **never** replaced by 0 and never imputed.
2. Columns that are entirely NaN for a mineral are dropped for that mineral. Rows with no data at all are dropped.
3. MCS codes like `W` (withheld), `NA`, `--`, `XX` become NaN, and `W` is flagged.
4. A coverage table shows how complete each series is, so you can decide what is usable.
5. Optional: gap-filling of short gaps (<= 2 years), only in a separate, flagged copy for plotting.

In [1]:
import re
from pathlib import Path
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 220)

# Locate the repo root (the folder that contains Raw_Data) by walking up from the notebook's folder.
def find_root(start: Path = Path.cwd()) -> Path:
    for p in [start, *start.parents]:
        if (p / "Raw_Data").is_dir():
            return p
    raise FileNotFoundError("Could not find a 'Raw_Data' folder above " + str(start))

ROOT = find_root()
DATA_DIR = ROOT / "Raw_Data"
MCS_FILE = DATA_DIR / "MCS2026_Commodities_Data.csv"
OUT_DIR = ROOT / "Project_Directory" / "Supply_demand"
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("Reading from:", DATA_DIR)
print("Saving to:   ", OUT_DIR)

ALIASES = {"germanum": "germanium"}      # the germanium file is spelled 'germanum' on disk

MINERALS = ["cobalt", "gallium", "germanium", "lithium", "silicon", "tantalum", "tin", "tungsten"]

# Units the ds140 files are assumed to use (the columns have no unit field).
# VERIFY against each file's header / the USGS DS140 documentation, then edit.
DS140_UNIT = {m: "metric tons" for m in MINERALS}
DS140_UNIT.update({"gallium": "kilograms", "germanium": "kilograms"})   # <- check these two

Reading from: /home/rameshp/vscode/Semiconductors-DSS/Raw_Data
Saving to:    /home/rameshp/vscode/Semiconductors-DSS/Project_Directory/Supply_demand


## 1. Load the ds140 files

In [4]:
import csv

CANON = ["Year", "Primary production", "Secondary production", "Mine shipments", "Government shipments",
         "Imports", "Exports", "Stocks", "Reported consumption", "Apparent consumption",
         "Unit value ($/t)", "Unit value (98$/t)", "World mine production", "World refinery production"]
_norm = lambda x: re.sub(r"\s+", " ", str(x).replace("\ufeff", "").replace("\xa0", " ")).strip().lower()
CANON_MAP = {_norm(c): c for c in CANON}

def rows_to_df(rows, name):
    """Find the header row (the one containing 'Year'), skipping any title rows above it."""
    for i, r in enumerate(rows):
        if any(_norm(c) == "year" for c in r):
            header = [CANON_MAP.get(_norm(c), str(c).replace("\ufeff", "").strip()) for c in r]
            body = [list(x) + [None] * (len(header) - len(x)) for x in rows[i + 1:]]
            body = [x[:len(header)] for x in body]
            df = pd.DataFrame(body, columns=header)
            return df.loc[:, [c for c in df.columns if c not in ("", "nan", "None")]]
    raise ValueError(f"{name}: no header row containing 'Year'. First rows seen: {rows[:4]}")

def read_any(p: Path) -> pd.DataFrame:
    if p.suffix.lower() in (".xlsx", ".xls"):
        raw = pd.read_excel(p, header=None, dtype=object)
        rows = raw.where(raw.notna(), None).values.tolist()
    else:
        with open(p, encoding="utf-8-sig", errors="replace", newline="") as f:   # utf-8-sig strips the BOM
            rows = list(csv.reader(f, skipinitialspace=True))
    return rows_to_df(rows, p.name)

MISSING_TOKENS = {"", "w", "na", "n/a", "nan", "--", "-", "xx", "(1)", "(2)", "(3)", "withheld", "e"}

def to_num(x):
    """Parse '1,234', '<1', '12e', '(3)' ... to float. Withheld/unknown -> NaN."""
    if pd.isna(x):
        return np.nan
    if isinstance(x, (int, float, np.number)):
        return float(x)
    s = str(x).strip().replace(",", "").replace("$", "")
    if s.lower() in MISSING_TOKENS:
        return np.nan
    m = re.search(r"-?\d+(?:\.\d+)?(?:[eE][+-]?\d+)?", s)
    return float(m.group(0)) if m else np.nan

frames = []
for p in sorted(DATA_DIR.glob("ds140-*.*")):
    m = re.match(r"ds140-(.+)-(\d{4})$", p.stem)
    if not m or p.suffix.lower() not in (".csv", ".xlsx", ".xls"):
        continue
    mineral, last_year = m.group(1).lower(), int(m.group(2))
    mineral = ALIASES.get(mineral, mineral)
    df = read_any(p)
    df = df.loc[:, ~df.columns.astype(str).str.startswith("Unnamed")]   # trailing comma -> empty column
    df.columns = [str(c).strip() for c in df.columns]
    df.insert(0, "mineral", mineral)
    df["file_last_year"] = last_year
    frames.append(df)
    print(f"{p.name:32s} rows={len(df):4d}  cols={len(df.columns)}  -> {list(df.columns)[:4]} ...")

missing = set(MINERALS) - {f["mineral"].iloc[0] for f in frames}
assert not missing, f"Missing ds140 files for: {missing}"
ds = pd.concat(frames, ignore_index=True, sort=False)
ds.head()

ds140-cobalt-2021.csv            rows= 127  cols=16  -> ['mineral', 'Year', 'Primary production', 'Secondary production'] ...
ds140-gallium-2021.csv           rows=  86  cols=11  -> ['mineral', 'Year', 'Production', 'Imports'] ...
ds140-germanum-2022.csv          rows=  86  cols=11  -> ['mineral', 'Year', 'Production', 'Imports'] ...
ds140-lithium-2021.csv           rows= 125  cols=12  -> ['mineral', 'Year', 'Production', 'Imports'] ...
ds140-silicon-2022.csv           rows= 103  cols=12  -> ['mineral', 'Year', 'Production', 'Shipments'] ...
ds140-tantalum-2022.csv          rows=  76  cols=14  -> ['mineral', 'Year', 'Mine production', 'Secondary production'] ...
ds140-tin-2021.csv               rows= 140  cols=14  -> ['mineral', 'Year', 'Primary production', 'Secondary production'] ...
ds140-tungsten-2019.csv          rows= 126  cols=14  -> ['mineral', 'Year', 'Primary production', 'Secondary production'] ...


,mineral,Year,Primary production,Secondary production,Mine shipments,Government shipments,Imports,Exports,Stocks,Reported consumption,Apparent consumption,Unit value ($/t),Unit value (98$/t),World mine production,World refinery production,file_last_year,Production,World production,Estimated consumption,World production (gross weight),World production (lithium content),World production (lithium carbonate equivalent),Shipments,Mine production,Producer shipments,Government stocks,Industry stocks
0,cobalt,1900,NA,NA,NA,0,20,NA,NA,NA,20,"4,930","97,000",NA,NA,2021,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,cobalt,1901,NA,NA,NA,0,20,NA,NA,NA,20,"5,590","110,000",180,NA,2021,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,cobalt,1902,NA,NA,NA,0,30,NA,NA,NA,30,"5,600","106,000",540,NA,2021,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,cobalt,1903,NA,NA,NA,0,20,NA,NA,NA,20,"6,050","110,000",640,NA,2021,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,cobalt,1904,NA,NA,NA,0,10,NA,NA,NA,10,"6,210","113,000",540,NA,2021,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 2. Clean ds140: years, numerics, NaN handling

In [5]:
VALUE_COLS = [c for c in ds.columns if c not in ("mineral", "Year", "file_last_year")]

ds["year"] = pd.to_numeric(ds["Year"].astype(str).str.extract(r"(\d{4})")[0], errors="coerce")
ds = ds.dropna(subset=["year"]).copy()
ds["year"] = ds["year"].astype(int)
for c in VALUE_COLS:
    ds[c] = ds[c].map(to_num)

# duplicate years within a mineral: keep the last row
ds = ds.drop_duplicates(subset=["mineral", "year"], keep="last")

# drop columns that are entirely NaN for a mineral, and rows with no data at all
clean = []
for mineral, g in ds.groupby("mineral"):
    g = g.copy()
    keep = [c for c in VALUE_COLS if g[c].notna().any()]
    g = g[["mineral", "year"] + keep]
    g = g.dropna(subset=keep, how="all")
    clean.append(g)
ds_wide = pd.concat(clean, ignore_index=True, sort=False).sort_values(["mineral", "year"])

# NaN coverage (% of years with a value) per mineral and column, between each mineral's first and last year
cols = [c for c in VALUE_COLS if c in ds_wide.columns]
coverage = ds_wide.groupby("mineral")[cols].agg(lambda s: round(s.notna().mean() * 100))
years = ds_wide.groupby("mineral")["year"].agg(first="min", last="max")
print("Share of years with data (%); NaN = column not present for that mineral")
display(years.join(coverage))

Share of years with data (%); NaN = column not present for that mineral


,first,last,Primary production,Secondary production,Mine shipments,Government shipments,Imports,Exports,Stocks,Reported consumption,Apparent consumption,Unit value ($/t),Unit value (98$/t),World mine production,World refinery production,Production,World production,Estimated consumption,World production (gross weight),World production (lithium content),World production (lithium carbonate equivalent),Shipments,Mine production,Producer shipments,Government stocks,Industry stocks
mineral,,,,,,,,,,,,,,,,,,,,,,,,,,
cobalt,1900,2021,78,65,50,80,100,66,60,65,100,100,100,99,44,0,0,0,0,0,0,0,0,0,0,0
gallium,1943,2021,0,0,0,0,75,9,65,0,66,100,100,0,0,58,62,0,0,0,0,0,0,0,0,0
germanium,1945,2022,0,0,0,0,88,28,47,0,85,100,100,0,0,72,82,0,0,0,0,0,0,0,0,0
lithium,1900,2021,0,0,0,0,51,42,0,0,0,58,58,0,0,43,0,100,80,18,18,0,0,0,0,0
silicon,1923,2022,0,0,0,0,100,59,59,0,67,100,100,0,0,67,59,0,0,0,0,62,0,0,0,0
tantalum,1964,2022,0,66,0,100,100,100,32,0,100,100,100,0,0,0,92,0,0,0,0,0,81,39,0,0
tin,1900,2021,67,68,0,65,100,88,0,0,100,100,100,0,0,0,96,0,0,0,0,0,0,0,34,75
tungsten,1900,2019,86,48,0,86,92,86,68,53,94,100,100,0,0,0,96,0,0,0,0,0,0,0,0,0


## 3. Standardise ds140 to a tidy table (metric tons)

`Unit value` columns are prices, not quantities, so they go in their own table. Reserves are not in ds140, only in the MCS file.

In [6]:
COLMAP = {   # ds140 column -> (metric, detail, country)
    "Primary production":      ("production",  "primary",            "United States"),
    "Secondary production":    ("production",  "secondary",          "United States"),
    "Mine shipments":          ("shipments",   "mine",               "United States"),
    "Government shipments":    ("shipments",   "government",         "United States"),
    "Imports":                 ("imports",     "imports",            "United States"),
    "Exports":                 ("exports",     "exports",            "United States"),
    "Stocks":                  ("stocks",      "stocks",             "United States"),
    "Reported consumption":    ("consumption", "reported",           "United States"),
    "Apparent consumption":    ("consumption", "apparent",           "United States"),
    "World mine production":   ("production",  "world mine",         "World"),
    "World refinery production": ("production", "world refinery",    "World"),
}
PRICE_COLS = ["Unit value ($/t)", "Unit value (98$/t)"]

def unit_factor(unit):
    """Factor to convert `unit` to metric tons. None if it is not a mass unit."""
    u = str(unit).lower()
    if "kilogram" in u or u.strip() in ("kg", "kilograms"):
        base = 1e-3
    elif "short ton" in u:
        base = 0.90718474
    elif "metric ton" in u or "tonne" in u or u.strip() in ("t", "mt"):
        base = 1.0
    else:
        return None
    if "million" in u:
        base *= 1e6
    elif "thousand" in u:
        base *= 1e3
    return base

rows = []
for col, (metric, detail, country) in COLMAP.items():
    if col not in ds_wide.columns:
        continue
    t = ds_wide[["mineral", "year", col]].dropna(subset=[col]).rename(columns={col: "value_raw"})
    t["unit_raw"] = t["mineral"].map(DS140_UNIT)
    t["factor"] = t["unit_raw"].map(unit_factor)
    t["value_t"] = t["value_raw"] * t["factor"]
    t["metric"], t["detail"], t["country"] = metric, detail, country
    t["source"], t["flag"] = "ds140", ""
    rows.append(t)
ds_tidy = pd.concat(rows, ignore_index=True)[
    ["mineral", "country", "year", "metric", "detail", "value_t", "value_raw", "unit_raw", "source", "flag"]]

prices = ds_wide[["mineral", "year"] + [c for c in PRICE_COLS if c in ds_wide.columns]]
ds_tidy.head()

,mineral,country,year,metric,detail,value_t,value_raw,unit_raw,source,flag
0,cobalt,United States,1910,production,primary,0.0,0.0,metric tons,ds140,
1,cobalt,United States,1911,production,primary,0.0,0.0,metric tons,ds140,
2,cobalt,United States,1912,production,primary,0.0,0.0,metric tons,ds140,
3,cobalt,United States,1913,production,primary,0.0,0.0,metric tons,ds140,
4,cobalt,United States,1914,production,primary,0.0,0.0,metric tons,ds140,


## 4. Load the MCS 2026 commodities file

In [7]:
mcs = pd.read_csv(MCS_FILE, encoding="cp1252", dtype=str, low_memory=False)
mcs.columns = [c.strip() for c in mcs.columns]
mcs["mineral"] = mcs["Commodity"].str.strip().str.lower()
mcs = mcs[mcs["mineral"].isin(MINERALS)].copy()
print("Rows for our 8 minerals:", len(mcs))

# Inspect before trusting the mapping below
print("\nStatistics values:")
print(mcs.groupby(["Statistics"]).size().sort_values(ascending=False).to_string())
print("\nUnits:")
print(mcs["Unit"].value_counts().to_string())

Rows for our 8 minerals: 810

Statistics values:
Statistics
Production                  289
Import                       75
Import sources 2021-2024     75
Export                       70
Reserves                     70
Price                        65
Consumption                  50
Net import reliance          50
Stock                        30
Shipment                     25
Capacity                      6
Employment                    5

Units:
Unit
metric tons                        410
percent                            125
thousand metric tons               107
kilograms                           98
cents per pound                     25
dollars per kilogram                20
dollars per pound                   10
dollars per metric ton               5
number                               5
dollars per dry metric ton unit      5


In [8]:
def classify(stat, detail):
    s = f"{stat} {detail}".lower()
    if "reserve" in s:      return "reserves"
    if "consumption" in s:  return "consumption"
    if "import" in s:       return "imports"
    if "export" in s:       return "exports"
    if "stock" in s:        return "stocks"
    if "production" in s or "mine" in s: return "production"
    return None   # price, employment, net import reliance ... not needed

mcs["metric"] = [classify(a, b) for a, b in zip(mcs["Statistics"].fillna(""), mcs["Statistics_detail"].fillna(""))]
mcs = mcs.dropna(subset=["metric"]).copy()

mcs["year"] = pd.to_numeric(mcs["Year"].astype(str).str.extract(r"(\d{4})")[0], errors="coerce")
mcs = mcs.dropna(subset=["year"]); mcs["year"] = mcs["year"].astype(int)

mcs["value_raw"] = mcs["Value"].map(to_num)
mcs["withheld"] = mcs["Value"].astype(str).str.strip().str.upper().eq("W")
mcs["factor"] = mcs["Unit"].map(unit_factor)
mcs["value_t"] = mcs["value_raw"] * mcs["factor"]

mcs["flag"] = ""
mcs.loc[mcs["withheld"], "flag"] = "withheld"
mcs.loc[mcs["factor"].isna(), "flag"] = "non-mass unit (not converted)"

mcs_tidy = pd.DataFrame({
    "mineral": mcs["mineral"], "country": mcs["Country"].str.strip(), "year": mcs["year"],
    "metric": mcs["metric"], "detail": mcs["Statistics_detail"].fillna(mcs["Statistics"]),
    "value_t": mcs["value_t"], "value_raw": mcs["value_raw"], "unit_raw": mcs["Unit"],
    "source": "MCS2026", "flag": mcs["flag"],
})
# drop NaN values (kept as flags only for withheld rows so you can see what was suppressed)
mcs_tidy = mcs_tidy[mcs_tidy["value_t"].notna() | (mcs_tidy["flag"] == "withheld")]
print(mcs_tidy["flag"].value_counts(dropna=False).to_string())
mcs_tidy.head()

flag
            461
withheld     48


,mineral,country,year,metric,detail,value_t,value_raw,unit_raw,source,flag
1788,cobalt,United States,2021,production,Production: Mine,650.0,650.0,metric tons,MCS2026,
1789,cobalt,United States,2021,production,Production: Secondary,1800.0,1800.0,metric tons,MCS2026,
1790,cobalt,United States,2021,consumption,Imports for consumption,9790.0,9790.0,metric tons,MCS2026,
1791,cobalt,United States,2021,exports,Exports,4930.0,4930.0,metric tons,MCS2026,
1792,cobalt,United States,2021,consumption,Consumption (includes secondary): Estimated,7270.0,7270.0,metric tons,MCS2026,


## 5. Combine and check

Production, consumption and trade may appear in both sources for overlapping years. The `source` column is kept so you can compare. For a single series, use **ds140 up to its last year and MCS after**, as in the helper below.

In [9]:
tidy = pd.concat([ds_tidy, mcs_tidy], ignore_index=True)
tidy = tidy[(tidy["value_t"].notna()) | (tidy["flag"] == "withheld")]

def series(mineral, metric, country="United States", detail_contains=None):
    """One series per year: ds140 for years it covers, MCS for later years."""
    s = tidy[(tidy.mineral == mineral) & (tidy.metric == metric) & (tidy.country == country)]
    if detail_contains:
        s = s[s.detail.str.contains(detail_contains, case=False, na=False)]
    s = s.dropna(subset=["value_t"]).sort_values(["year", "source"])   # 'MCS2026' < 'ds140' alphabetically
    last_ds = s.loc[s.source == "ds140", "year"].max()
    s = s[((s.source == "ds140")) | (s.year > (last_ds if pd.notna(last_ds) else -1))]
    return s.groupby("year")["value_t"].sum(min_count=1)

# sanity: how many values per mineral / metric / source
display(tidy.pivot_table(index=["mineral", "metric"], columns="source", values="value_t", aggfunc="count", fill_value=0))

source                 MCS2026  ds140
mineral   metric                     
cobalt    consumption       15    201
          exports            5     81
          imports            0    122
          production        38    349
          reserves          14      0
          shipments          0    159
          stocks            10     73
gallium   consumption       15     52
          exports            0      7
          imports            5     59
          production        13      0
          stocks             5     51
germanium consumption       11     66
          exports           10     22
          imports            0     69
          stocks             0     37
lithium   consumption        5      0
          exports            5     51
          imports            0     62
          production        20      0
          reserves          12      0
silicon   consumption       10     67
          exports           10     59
          imports            0    100
          production        58      0
          stocks             3     59
tantalum  consumption       10     59
          exports            5     59
          imports            0     59
          production        24     39
          reserves           4      0
          shipments          0     59
          stocks             1     19
tin       consumption       20    122
          exports           15    107
          imports            0    122
          production        42    165
          reserves          12      0
          shipments          0     79
          stocks             6      0
tungsten  consumption       10    177
          exports           10    103
          imports            0    110
          production        25    161
          reserves          10      0
          shipments          0    103
          stocks             3     81

## 6. US supply and demand summary (wide)

In [10]:
def pick_consumption(g):
    """Apparent consumption if present, else reported; record which one was used."""
    a = g.get("Apparent consumption"); r = g.get("Reported consumption")
    val = a if a is not None else pd.Series(np.nan, index=g.index)
    if r is not None:
        val = val.fillna(r)
    return val

summary = ds_wide.copy()
summary = summary.assign(
    production_total=summary[[c for c in ["Primary production", "Secondary production"] if c in summary]].sum(axis=1, min_count=1),
    consumption=pick_consumption(summary),
)
summary["unit"] = summary["mineral"].map(DS140_UNIT)
# convert to metric tons
fac = summary["unit"].map(unit_factor)
for c in [c for c in summary.columns if c in COLMAP or c in ("production_total", "consumption")]:
    summary[c] = summary[c] * fac
summary["unit"] = "metric tons"

# latest reserves (MCS) per mineral and country
reserves = (tidy[(tidy.metric == "reserves") & tidy.value_t.notna()]
            .sort_values("year").groupby(["mineral", "country"]).tail(1)
            [["mineral", "country", "year", "value_t", "detail"]]
            .rename(columns={"value_t": "reserves_t"}))
display(summary.tail(10)); display(reserves.head(20))

,mineral,year,Primary production,Secondary production,Mine shipments,Government shipments,Imports,Exports,Stocks,Reported consumption,Apparent consumption,Unit value ($/t),Unit value (98$/t),World mine production,World refinery production,Production,World production,Estimated consumption,World production (gross weight),World production (lithium content),World production (lithium carbonate equivalent),Shipments,Mine production,Producer shipments,Government stocks,Industry stocks,production_total,consumption,unit
792,tungsten,2010,NaN,5680.0,NaN,2060.0,12400.0,4620.0,1310.0,11000.0,15500.0,27000.0,20200.0,NaN,NaN,NaN,68400.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5680.0,15500.0,metric tons
793,tungsten,2011,NaN,11000.0,NaN,1220.0,13300.0,7260.0,1240.0,12300.0,18000.0,46700.0,33900.0,NaN,NaN,NaN,73900.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,11000.0,18000.0,metric tons
794,tungsten,2012,NaN,9180.0,NaN,1780.0,11700.0,6770.0,1360.0,11400.0,14900.0,56700.0,40200.0,NaN,NaN,NaN,77700.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,9180.0,14900.0,metric tons
795,tungsten,2013,NaN,8600.0,NaN,2100.0,12100.0,7710.0,1420.0,10700.0,14600.0,46600.0,32600.0,NaN,NaN,NaN,79400.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,8600.0,14600.0,metric tons
796,tungsten,2014,NaN,NaN,NaN,282.0,12900.0,6730.0,1350.0,11600.0,NaN,45200.0,31100.0,NaN,NaN,NaN,82100.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,11600.0,metric tons
797,tungsten,2015,NaN,NaN,NaN,0.0,10200.0,3760.0,1030.0,11100.0,NaN,37800.0,26000.0,NaN,NaN,NaN,83700.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,11100.0,metric tons
798,tungsten,2016,0.0,NaN,NaN,0.0,9880.0,3380.0,1070.0,10400.0,NaN,25000.0,17000.0,NaN,NaN,NaN,78400.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,10400.0,metric tons
799,tungsten,2017,0.0,NaN,NaN,1460.0,13700.0,3550.0,1170.0,11300.0,NaN,35000.0,23300.0,NaN,NaN,NaN,81400.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,11300.0,metric tons
800,tungsten,2018,0.0,NaN,NaN,1180.0,14400.0,3500.0,1080.0,11800.0,NaN,45900.0,29800.0,NaN,NaN,NaN,82200.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,11800.0,metric tons
801,tungsten,2019,0.0,NaN,NaN,661.0,13900.0,3370.0,1050.0,11800.0,NaN,38800.0,24700.0,NaN,NaN,NaN,83800.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,11800.0,metric tons


,mineral,country,year,reserves_t,detail
3438,cobalt,United States,2025,70000.0,Reserves
3704,tantalum,China,2025,240000.0,Reserves
3705,tantalum,Russia,2025,150.0,Reserves
3789,tin,Australia,2025,570000.0,Reserves
3790,tin,Bolivia,2025,400000.0,Reserves
3791,tin,Brazil,2025,700000.0,Reserves
3792,tin,Burma,2025,700000.0,Reserves
3793,tin,China,2025,1200000.0,Reserves
3794,tin,Congo (Kinshasa),2025,91000.0,Reserves
3795,tin,Indonesia,2025,1400000.0,Reserves


## 7. Optional: gap-filled copy for plotting only

Short gaps (<= 2 consecutive years) are linearly interpolated, and every filled value is flagged. Never use this for statistics you report as measured.

In [11]:
def fill_short_gaps(df, cols, limit=2):
    out = df.sort_values(["mineral", "year"]).copy()
    for c in cols:
        filled = out.groupby("mineral")[c].transform(lambda s: s.interpolate(limit=limit, limit_area="inside"))
        out[c + "_filled"] = out[c].isna() & filled.notna()
        out[c] = filled
    return out

plot_cols = [c for c in ["production_total", "consumption"] if c in summary.columns]
summary_plot = fill_short_gaps(summary, plot_cols)
print(summary_plot[[c + "_filled" for c in plot_cols]].sum())

production_total_filled    9
consumption_filled         2
dtype: int64


## 8. Save

In [19]:
FINAL_CSV = OUT_DIR / "minerals_supply_demand.csv"

final = (tidy[tidy["value_t"].notna()]                 # drop rows with no value (the withheld ones)
             .drop(columns="flag")
             .sort_values(["mineral", "metric", "country", "year", "source"])
             .reset_index(drop=True))
final.to_csv(FINAL_CSV, index=False)

# the supporting tables go in the same folder
summary.to_csv(OUT_DIR / "minerals_us_summary_wide.csv", index=False)
reserves.to_csv(OUT_DIR / "minerals_reserves_latest.csv", index=False)
coverage.to_csv(OUT_DIR / "ds140_nan_coverage_pct.csv")
prices.to_csv(OUT_DIR / "ds140_prices.csv", index=False)

print("Saved final CSV:", FINAL_CSV.resolve())
print("Shape:", final.shape)

Saved final CSV: /home/rameshp/vscode/Semiconductors-DSS/Project_Directory/Supply_demand/minerals_supply_demand.csv
Shape: (3831, 9)


## 9. First 50 rows of the saved CSV

In [20]:
check = pd.read_csv(FINAL_CSV)          # read back from disk to show exactly what was saved
with pd.option_context("display.max_rows", 60, "display.max_columns", None, "display.width", 250):
    display(check.head(50))
print(f"Shape: {check.shape[0]} rows, {check.shape[1]} columns")
check.info()


,mineral,country,year,metric,detail,value_t,value_raw,unit_raw,source
0,cobalt,United States,1900,consumption,apparent,20.0,20.0,metric tons,ds140
1,cobalt,United States,1901,consumption,apparent,20.0,20.0,metric tons,ds140
2,cobalt,United States,1902,consumption,apparent,30.0,30.0,metric tons,ds140
3,cobalt,United States,1903,consumption,apparent,20.0,20.0,metric tons,ds140
4,cobalt,United States,1904,consumption,apparent,10.0,10.0,metric tons,ds140
5,cobalt,United States,1905,consumption,apparent,20.0,20.0,metric tons,ds140
6,cobalt,United States,1906,consumption,apparent,10.0,10.0,metric tons,ds140
7,cobalt,United States,1907,consumption,apparent,20.0,20.0,metric tons,ds140
8,cobalt,United States,1908,consumption,apparent,70.0,70.0,metric tons,ds140
9,cobalt,United States,1909,consumption,apparent,5.0,5.0,metric tons,ds140


Shape: 3831 rows, 9 columns
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3831 entries, 0 to 3830
Data columns (total 9 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   mineral    3831 non-null   object 
 1   country    3831 non-null   object 
 2   year       3831 non-null   int64  
 3   metric     3831 non-null   object 
 4   detail     3831 non-null   object 
 5   value_t    3831 non-null   float64
 6   value_raw  3831 non-null   float64
 7   unit_raw   3831 non-null   object 
 8   source     3831 non-null   object 
dtypes: float64(2), int64(1), object(6)
memory usage: 269.5+ KB
